# VGG-Lite Hardware Accelerator — Final Evaluation (10,000 images)

**Final project — Deployment of Convolutional Neural Networks on FPGA**

Aviv Holtzman & Dekel Zur Shaday · Ruppin Academic Center, Dept. of Electrical & Electronics Engineering
Advisors: Dr. Rafi Cohen, Dr. Benjamin Gur Salomon

---

## What this notebook measures

The notebook runs on the PYNQ-Z2 board and produces every result reported in the
project, written to `results_10k.json`:

1. **Equivalence** — the accelerator's output compared element by element with the
   compiled model.
2. **Accuracy** — on the full CIFAR-10 test set (10,000 images). The reference data
   is read in chunks of 1,000, because the board has 512 MB of RAM and the full
   `acc_in` array alone is 164 MB.
3. **Timing** — latency and throughput, for the accelerated layers and for the whole
   system.

Two speed-ups are reported, because they answer two different questions:

| Question | Comparison |
|---|---|
| How much faster is the accelerator than the same layers in software? | software conv2→pool→conv3→conv4→pool  vs  accelerator |
| How much faster is the *system*? | whole network in software  vs  software head + accelerator + software tail |

The accelerator executes only conv2, conv3 and conv4. In a real run the ARM also
computes conv1 and the quantiser before it, and the dequantiser, average-pool and
fully-connected layer after it, so the system speed-up counts those stages too.

443,648 of the 6,341,888 MACs per image (7.0 %) stay in software. If run time were
proportional to MACs, Amdahl's law would cap the system speed-up at about 14.3x —
the value printed below as "Amdahl ceiling". The measured times show that the
software stages take far longer than their share of the MACs, so the ceiling based
on measured time is much lower (3.38x; see the project book, section 6.6).

## System architecture (HW/SW co-design)

| Stage | Executed on | Operations |
|---|---|---|
| Normalisation + conv1 + quantisation | **PS (ARM Cortex-A9)** | `Transpose → Im2Col → MatMul → MultiThreshold` |
| conv2 → MaxPool → conv3 → conv4 → MaxPool | **PL (Zynq-7020 FPGA)** | 3x MVAU, 3x FMPadding, 3x SlidingWindow, 2x MaxPool |
| Dequantise + AvgPool + Flatten + FC | **PS (ARM Cortex-A9)** | `Transpose → Mul → AveragePool → Flatten → MatMul → Mul → Add` |

The partition boundary was chosen by the FINN compiler, not by us: the network
input is float32 (the Brevitas model has no input quantiser), so FINN cannot
build thresholds for conv1; and `nn.AvgPool2d` is not a supported quantised
operator, so the classifier is excluded. FINN extracts one *contiguous*
subgraph, which is why the two excluded layers are necessarily at the ends.

Accelerator interface: input `(1,32,32,16) UINT8`, output `(1,8,8,32) UINT8`.

## 1. Locate the driver, the bitstream and the reference chunks

In [1]:
import sys, os, glob, json, time
import numpy as np

# --- locate the FINN driver and the bitstream ---
drv = sorted(glob.glob("**/driver.py", recursive=True))
bit = sorted(glob.glob("**/finn-accel.bit", recursive=True))
assert drv and bit, "driver.py or finn-accel.bit not found below the notebook folder"

DRIVER_DIR = os.path.dirname(drv[0])
BITFILE    = bit[0]
WEIGHT_DIR = os.path.join(DRIVER_DIR, "runtime_weights")
sys.path.insert(0, DRIVER_DIR)

from driver_base import FINNExampleOverlay
from driver import io_shape_dict

# --- locate the chunked reference data ---
REF_DIR = "reference_10k"
CHUNKS  = sorted(glob.glob(os.path.join(REF_DIR, "ref_chunk_*.npz")))
TAIL    = os.path.join(REF_DIR, "ref_tail_params.npz")
assert CHUNKS, f"no reference chunks found in {REF_DIR}/"
assert os.path.exists(TAIL), f"{TAIL} not found"
REF_1K = "reference_data.npz"        # the original reference, the only one with acc_out
assert os.path.exists(REF_1K), f"{REF_1K} not found (needed for the equivalence check)"

print(f"driver      : {DRIVER_DIR}")
print(f"bitstream   : {BITFILE}")
print(f"ref chunks  : {len(CHUNKS)} files, "
      f"{sum(os.path.getsize(c) for c in CHUNKS)/1e6:.0f} MB  -> accuracy")
print(f"equivalence : {REF_1K}  -> expected feature maps")
print()
print("FINN accelerator I/O specification")
print("=" * 58)
for k, v in io_shape_dict.items():
    print(f"  {k:26s} : {v}")

driver      : deploy/build_vgg_lite_int8_bitfile_final9283rce6/deploy/driver
bitstream   : deploy/build_vgg_lite_int8_bitfile_final9283rce6/bitfile/finn-accel.bit
ref chunks  : 10 files, 132 MB  -> accuracy
equivalence : reference_data.npz  -> expected feature maps

FINN accelerator I/O specification
  idt                        : [UINT8]
  odt                        : [UINT8]
  ishape_normal              : [(1, 32, 32, 16)]
  oshape_normal              : [(1, 8, 8, 32)]
  ishape_folded              : [(1, 32, 32, 1, 16)]
  oshape_folded              : [(1, 8, 8, 1, 32)]
  ishape_packed              : [(1, 32, 32, 1, 16)]
  oshape_packed              : [(1, 8, 8, 1, 32)]
  input_dma_name             : ['idma0']
  output_dma_name            : ['odma0']
  number_of_external_weights : 0
  num_inputs                 : 1
  num_outputs                : 1


## 2. Load the overlay

The design was synthesised with `synth_clk_period_ns = 20.0`, i.e. a **50 MHz** PL
clock. The driver defaults to 100 MHz, so `fclk_mhz` **must** be set explicitly —
running the design above the frequency it closed timing at would corrupt results
silently rather than fail loudly.

In [2]:
BATCH = 25   # kept small: FIFO depths were capped during synthesis

accel = FINNExampleOverlay(
    bitfile_name       = BITFILE,
    platform           = "zynq-iodma",
    io_shape_dict      = io_shape_dict,
    batch_size         = BATCH,
    fclk_mhz           = 50.0,
    runtime_weight_dir = WEIGHT_DIR,
)

print("Bitstream loaded into the PL.")
print(f"  input  : {accel.ishape_normal()}  {accel.idt()}")
print(f"  output : {accel.oshape_normal()}  {accel.odt()}")

Bitstream loaded into the PL.
  input  : (25, 32, 32, 16)  UINT8
  output : (25, 8, 8, 32)  UINT8


## 3. Two questions, measured separately

**Equivalence — is the hardware doing exactly what the compiled model does?**
The accelerator's output feature map is compared element by element against the
one the compiled graph produces on the host. This is a property of the
*hardware*, not of the dataset, so it does not need the whole test set: 1,000
images give 2,048,000 compared elements, which settles the question. It runs
against `reference_data.npz` — the only file that carries the
expected output feature maps.

**Accuracy — how often is the system right?**
This *does* depend on sample size, so it runs over all 10,000 images. The
chunked files carry the accelerator input and the labels; the software tail is
applied to the hardware's own feature maps, exactly as a deployed system would.
Chunks are processed one at a time, so peak memory stays around 30 MB.

In [3]:
tp   = np.load(TAIL)
perm = tuple(tp["perm"]); kern = int(tp["kern"])
mul0, mul1 = tp["mul0"], tp["mul1"]
fcW,  bias = tp["fcW"],  tp["bias"]

def classify(feat_nhwc):
    """Software tail: NHWC uint8 feature map -> class logits."""
    f = feat_nhwc.astype(np.float32).transpose(perm) * mul0     # -> NCHW, dequantised
    n, c, h, w = f.shape
    f = f.reshape(n, c, h // kern, kern, w // kern, kern).mean(axis=(3, 5))
    return (f.reshape(n, -1) @ fcW) * mul1 + bias

def run_hw(x):
    """Run the accelerator over x in batches of BATCH; return int32 feature maps."""
    accel.batch_size = BATCH
    out = []
    for s in range(0, len(x), BATCH):
        chunk = x[s:s + BATCH]
        k = len(chunk)
        if k < BATCH:                                   # pad the last partial batch
            pad = np.zeros((BATCH - k,) + chunk.shape[1:], chunk.dtype)
            chunk = np.concatenate([chunk, pad])
        out.append(accel.execute(chunk)[:k])
    return np.concatenate(out).astype(np.int32)

# ==========================================================================
# PART A -- equivalence, against the reference that carries expected outputs
# ==========================================================================
r1k = np.load(REF_1K)
acc_in_1k, acc_out_1k = r1k["acc_in"], r1k["acc_out"]
n_eq = len(acc_in_1k)

t0 = time.time()
hw1k = run_hw(acc_in_1k)
print(f"equivalence run: {n_eq} images in {time.time()-t0:.2f} s\n")

ref = acc_out_1k.astype(np.int32)
exact = 100 * (hw1k == ref).mean()
near  = 100 * (np.abs(hw1k - ref) <= 1).mean()
corr  = float(np.corrcoef(hw1k.ravel(), ref.ravel())[0, 1])
hw_lo, hw_hi = int(hw1k.min()), int(hw1k.max())

print("HW / SW EQUIVALENCE   (feature map 8x8x32, element-wise)")
print("=" * 62)
print(f"  images compared      : {n_eq:,}")
print(f"  elements compared    : {ref.size:,}")
print(f"  hardware value range : {hw_lo} .. {hw_hi}")
print(f"  reference range      : {ref.min()} .. {ref.max()}")
print(f"  exact match          : {exact:.4f} %")
print(f"  within +/- 1 LSB     : {near:.4f} %")
print(f"  correlation          : {corr:+.6f}")
print("=" * 62)
print("PASS" if exact > 99.9 else "MISMATCH - investigate before writing anything up")

del hw1k, ref, r1k, acc_in_1k, acc_out_1k

equivalence run: 1000 images in 8.32 s

HW / SW EQUIVALENCE   (feature map 8x8x32, element-wise)
  images compared      : 1,000
  elements compared    : 2,048,000
  hardware value range : 0 .. 255
  reference range      : 0 .. 255
  exact match          : 100.0000 %
  within +/- 1 LSB     : 100.0000 %
  correlation          : +1.000000
PASS


In [4]:
# ==========================================================================
# PART B -- accuracy over all 10,000 images, one chunk at a time
# ==========================================================================
all_pred, all_labels = [], []
t_start = time.time()

for ci, path in enumerate(CHUNKS):
    z = np.load(path)
    acc_in, labels_c = z["acc_in"], z["labels"]
    t0 = time.time()
    hw = run_hw(acc_in)
    all_pred.append(classify(hw).argmax(1))
    all_labels.append(labels_c)
    print(f"  chunk {ci:02d}  {len(acc_in):5d} images  {time.time()-t0:6.2f} s")
    del z, acc_in, hw

hw_pred = np.concatenate(all_pred)
labels  = np.concatenate(all_labels)
N       = len(labels)
acc_hw  = 100 * (hw_pred == labels).mean()

print(f"\nprocessed {N:,} images in {time.time()-t_start:.1f} s\n")
print("CLASSIFICATION ACCURACY")
print("=" * 62)
print(f"  HW/SW co-design system : {acc_hw:6.2f} %")
print(f"  correct                : {int((hw_pred == labels).sum()):,} / {N:,}")
print("=" * 62)

CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
print("\nPer-class accuracy:")
for i, name in enumerate(CLASSES):
    m = labels == i
    print(f"  {name:12s} {100*(hw_pred[m] == labels[m]).mean():6.2f} %   ({int(m.sum())} images)")

  chunk 00   1000 images    8.51 s
  chunk 01   1000 images    8.50 s
  chunk 02   1000 images    8.50 s
  chunk 03   1000 images    8.50 s
  chunk 04   1000 images    8.50 s
  chunk 05   1000 images    8.50 s
  chunk 06   1000 images    8.50 s
  chunk 07   1000 images    8.51 s
  chunk 08   1000 images    8.51 s
  chunk 09   1000 images    8.50 s

processed 10,000 images in 97.5 s

CLASSIFICATION ACCURACY
  HW/SW co-design system :  70.29 %
  correct                : 7,029 / 10,000

Per-class accuracy:
  airplane      79.20 %   (1000 images)
  automobile    86.40 %   (1000 images)
  bird          55.80 %   (1000 images)
  cat           44.70 %   (1000 images)
  deer          60.60 %   (1000 images)
  dog           69.90 %   (1000 images)
  frog          80.70 %   (1000 images)
  horse         70.10 %   (1000 images)
  ship          77.30 %   (1000 images)
  truck         78.20 %   (1000 images)


## 4. Timing — the three parts of the pipeline, measured separately

`accel.execute()` measures **only** the FPGA stage. To state anything about the
system we must also measure what the ARM does before and after it.

* **Head** — conv1 followed by the quantiser that produces the UINT8 accelerator
  input. We hold the exported thresholds only implicitly, so the quantiser is
  timed against a correctly-shaped dummy threshold array. That over-states its
  cost slightly, which is the safe direction for a speed-up claim.
* **Accelerator** — `accel.execute()`, including DMA and driver overhead.
* **Tail** — dequantise, average-pool, flatten, fully-connected.

Timing is measured on a sample of images, not on all 10,000: latency per image
does not depend on how many images we run, and the software baseline at ~100 ms
per image would take 17 minutes for the full set with nothing gained.

In [5]:
sw = np.load("sw_weights.npz")

def conv2d(x, W, b, pad=1):
    """x (N,C,H,W) float32, W (O,C,k,k) -> (N,O,H,W).  im2col + BLAS matmul."""
    N, C, H, Wd = x.shape
    O, _, k, _  = W.shape
    xp = np.pad(x, ((0,0), (0,0), (pad,pad), (pad,pad)))
    OH, OW = H + 2*pad - k + 1, Wd + 2*pad - k + 1
    cols = np.empty((N, k*k*C, OH*OW), np.float32)
    for i in range(k):
        for j in range(k):
            cols[:, (i*k+j)*C:(i*k+j+1)*C, :] = xp[:, :, i:i+OH, j:j+OW].reshape(N, C, -1)
    Wm = W.transpose(0, 2, 3, 1).reshape(O, -1)
    return (np.matmul(Wm, cols) + b[None, :, None]).reshape(N, O, OH, OW)

def maxpool2(x):
    N, C, H, W = x.shape
    return x.reshape(N, C, H//2, 2, W//2, 2).max(axis=(3, 5))

# MultiThreshold: a per-channel search against 255 sorted thresholds.
# The exported threshold values are not available here, so a correctly shaped
# dummy array is used. This measures the operation's cost, not its result.
_TH = np.sort(np.random.rand(16, 255).astype(np.float32), axis=1)

def quantise(y):
    """(N,16,32,32) float32 -> (N,32,32,16) uint8, the accelerator's input form."""
    z = y.transpose(0, 2, 3, 1)
    out = np.empty(z.shape, np.uint8)
    for c in range(z.shape[-1]):
        out[..., c] = np.searchsorted(_TH[c], z[..., c]).astype(np.uint8)
    return out

def sw_head(x):
    """Normalised image -> accelerator input.  conv1 + ReLU + quantiser."""
    return quantise(np.maximum(conv2d(x, sw["W1"], sw["b1"]), 0))

def sw_middle(x1):
    """Exactly the layers the accelerator replaces: conv2 -> pool -> conv3 -> conv4 -> pool."""
    x = np.maximum(conv2d(x1, sw["W2"], sw["b2"]), 0)
    x = maxpool2(x)
    x = np.maximum(conv2d(x, sw["W3"], sw["b3"]), 0)
    x = np.maximum(conv2d(x, sw["W4"], sw["b4"]), 0)
    return maxpool2(x)

def sw_tail(x):
    """AvgPool 4x4 + flatten + fully-connected."""
    N, C, H, W = x.shape
    x = x.reshape(N, C, H//4, 4, W//4, 4).mean(axis=(3, 5))
    return x.reshape(N, -1) @ sw["Wf"].T + sw["bf"]

def sw_full(x):
    x = np.maximum(conv2d(x, sw["W1"], sw["b1"]), 0)
    x = np.maximum(conv2d(x, sw["W2"], sw["b2"]), 0)
    x = maxpool2(x)
    x = np.maximum(conv2d(x, sw["W3"], sw["b3"]), 0)
    x = np.maximum(conv2d(x, sw["W4"], sw["b4"]), 0)
    x = maxpool2(x)
    return sw_tail(x)

def timeit(fn, arg, reps, warmup=3):
    for _ in range(warmup):
        fn(arg)
    t = time.perf_counter()
    for _ in range(reps):
        fn(arg)
    return (time.perf_counter() - t) / reps * 1000      # ms per call

# --- rebuild the normalised input for the timing sample ---------------------
z0   = np.load(CHUNKS[0])
MEAN = np.array([0.4914, 0.4822, 0.4465], np.float32)
STD  = np.array([0.2023, 0.1994, 0.2010], np.float32)
imgs = z0["imgs"].astype(np.float32) / 255.0
xin  = ((imgs - MEAN) / STD).transpose(0, 3, 1, 2).astype(np.float32)
acc_in0 = z0["acc_in"]
print("timing sample ready:", xin.shape)

timing sample ready: (1000, 3, 32, 32)


In [6]:
print("LATENCY  — one image end to end")
print("=" * 62)

# ---- FPGA stage ------------------------------------------------------------
accel.batch_size = 1
one_acc = acc_in0[:1]
for _ in range(20):
    accel.execute(one_acc)
lat = []
for _ in range(200):
    t0 = time.perf_counter()
    accel.execute(one_acc)
    lat.append((time.perf_counter() - t0) * 1000)
lat = np.array(lat)
t_accel = lat.mean()
print(f"  accelerator (incl. DMA + driver) : {t_accel:7.2f} ms   "
      f"(median {np.median(lat):.2f}, std {lat.std():.2f})")

# ---- software stages -------------------------------------------------------
one_img  = xin[:1]
one_mid  = np.maximum(conv2d(one_img, sw["W1"], sw["b1"]), 0)   # conv1 output, NCHW
one_feat = sw_middle(one_mid)

t_head   = timeit(sw_head,   one_img,  20)
t_middle = timeit(sw_middle, one_mid,  10)
t_tail   = timeit(sw_tail,   one_feat, 50)
t_sw_all = timeit(sw_full,   one_img,  10)

print(f"  software head  (conv1 + quantiser): {t_head:7.2f} ms")
print(f"  software tail  (deq+pool+FC)      : {t_tail:7.2f} ms")
print(f"  software middle (what the FPGA does): {t_middle:7.2f} ms")
print(f"  software, whole network           : {t_sw_all:7.2f} ms")
print("-" * 62)

t_system = t_head + t_accel + t_tail
print(f"  SYSTEM with accelerator           : {t_system:7.2f} ms"
      f"   ({t_head:.2f} + {t_accel:.2f} + {t_tail:.2f})")
print("=" * 62)
print(f"  accelerator speed-up (same layers): x{t_middle/t_accel:5.2f}")
print(f"  SYSTEM speed-up (end to end)      : x{t_sw_all/t_system:5.2f}")
print(f"  Amdahl ceiling (7.0 % in software): x{1/(1-5898240/6341888):5.2f}")
print("=" * 62)

# consistency check: the parts should roughly add up to the whole
parts = t_head + t_middle + t_tail
print(f"\n  sanity: head+middle+tail = {parts:.2f} ms  vs  whole network "
      f"{t_sw_all:.2f} ms   ({100*parts/t_sw_all:.0f} %)")

LATENCY  — one image end to end
  accelerator (incl. DMA + driver) :   11.05 ms   (median 11.02, std 0.07)
  software head  (conv1 + quantiser):   20.35 ms
  software tail  (deq+pool+FC)      :    8.60 ms
  software middle (what the FPGA does):   75.37 ms
  software, whole network           :   97.96 ms
--------------------------------------------------------------
  SYSTEM with accelerator           :   40.00 ms   (20.35 + 11.05 + 8.60)
  accelerator speed-up (same layers): x 6.82
  SYSTEM speed-up (end to end)      : x 2.45
  Amdahl ceiling (7.0 % in software): x14.29

  sanity: head+middle+tail = 104.32 ms  vs  whole network 97.96 ms   (106 %)


In [7]:
print("THROUGHPUT  — pipeline kept full")
print("=" * 62)

THEORETICAL_MS = 393216 / 50e6 * 1000

accel.batch_size = BATCH
blk = acc_in0[:BATCH]
accel.execute(blk)
REPS = 10
t = time.perf_counter()
for _ in range(REPS):
    accel.execute(blk)
thr_accel = (time.perf_counter() - t) / (REPS * BATCH) * 1000

blk_img = xin[:BATCH]
sw_full(blk_img)
t = time.perf_counter()
for _ in range(3):
    sw_full(blk_img)
thr_sw = (time.perf_counter() - t) / (3 * BATCH) * 1000

blk_mid = np.maximum(conv2d(blk_img, sw["W1"], sw["b1"]), 0)
sw_middle(blk_mid)
t = time.perf_counter()
for _ in range(3):
    sw_middle(blk_mid)
thr_mid = (time.perf_counter() - t) / (3 * BATCH) * 1000

thr_head = timeit(sw_head, blk_img, 3) / BATCH
thr_tail = timeit(sw_tail, sw_middle(blk_mid), 5) / BATCH
thr_system = thr_head + thr_accel + thr_tail

print(f"  accelerator        : {thr_accel:6.2f} ms/img  ({1000/thr_accel:6.1f} img/s)")
print(f"  theoretical bound  : {THEORETICAL_MS:6.2f} ms/img")
print(f"  pipeline efficiency: {100*THEORETICAL_MS/thr_accel:6.1f} %")
print("-" * 62)
print(f"  software, whole net: {thr_sw:6.2f} ms/img  ({1000/thr_sw:6.1f} img/s)")
print(f"  system w/ accel    : {thr_system:6.2f} ms/img  ({1000/thr_system:6.1f} img/s)")
print("=" * 62)
print(f"  accelerator speed-up (same layers): x{thr_mid/thr_accel:5.2f}")
print(f"  SYSTEM speed-up                   : x{thr_sw/thr_system:5.2f}")
print("=" * 62)

print("\nDRIVER OVERHEAD BREAKDOWN")
for k, v in accel.throughput_test().items():
    print(f"  {k:38s} {v}")

THROUGHPUT  — pipeline kept full
  accelerator        :   8.15 ms/img  ( 122.7 img/s)
  theoretical bound  :   7.86 ms/img
  pipeline efficiency:   96.5 %
--------------------------------------------------------------
  software, whole net:  57.43 ms/img  (  17.4 img/s)
  system w/ accel    :  19.59 ms/img  (  51.1 img/s)
  accelerator speed-up (same layers): x 6.28
  SYSTEM speed-up                   : x 2.93

DRIVER OVERHEAD BREAKDOWN
  runtime[ms]                            198.1041431427002
  throughput[images/s]                   126.19625013088076
  DRAM_in_bandwidth[MB/s]                2.06759936214435
  DRAM_out_bandwidth[MB/s]               0.25844992026804375
  fclk[mhz]                              50.0
  batch_size                             25
  fold_input[ms]                         0.12254714965820312
  pack_input[ms]                         0.09250640869140625
  copy_input_data_to_device[ms]          2.7544498443603516
  copy_output_data_from_device[ms]       0.514268

## 5. Summary and results file

In [8]:
MACS_HW, MACS_TOTAL = 5_898_240, 6_341_888

results = {
    "n_images": int(N),
    "equivalence": {
        "n_images": int(n_eq),
        "elements_compared": int(n_eq * 8 * 8 * 32),
        "exact_match_pct": round(exact, 4),
        "within_1_lsb_pct": round(near, 4),
        "correlation": round(float(corr), 6),
        "hw_range": [hw_lo, hw_hi],
    },
    "accuracy": {
        "hw_system_pct": round(acc_hw, 2),
        "per_class": {CLASSES[i]: round(100*float((hw_pred[labels==i]==i).mean()), 2)
                      for i in range(10)},
    },
    "latency_ms": {
        "software_head_conv1_quant": round(float(t_head), 3),
        "accelerator": round(float(t_accel), 3),
        "software_tail": round(float(t_tail), 3),
        "system_total": round(float(t_system), 3),
        "software_middle_equivalent": round(float(t_middle), 3),
        "software_whole_network": round(float(t_sw_all), 3),
        "speedup_accelerator_same_layers": round(float(t_middle/t_accel), 2),
        "speedup_system": round(float(t_sw_all/t_system), 2),
    },
    "throughput_ms_per_image": {
        "accelerator": round(float(thr_accel), 3),
        "system_total": round(float(thr_system), 3),
        "software_whole_network": round(float(thr_sw), 3),
        "software_middle_equivalent": round(float(thr_mid), 3),
        "theoretical_bound": round(float(THEORETICAL_MS), 3),
        "pipeline_efficiency_pct": round(100*float(THEORETICAL_MS/thr_accel), 1),
        "speedup_accelerator_same_layers": round(float(thr_mid/thr_accel), 2),
        "speedup_system": round(float(thr_sw/thr_system), 2),
    },
    "compute": {
        "macs_in_hardware": MACS_HW,
        "macs_total": MACS_TOTAL,
        "hardware_share_pct": round(100*MACS_HW/MACS_TOTAL, 1),
        "amdahl_ceiling": round(1/(1-MACS_HW/MACS_TOTAL), 2),
        "gops_at_throughput": round(2*MACS_HW/(thr_accel/1000)/1e9, 2),
    },
    "resources": {"LUT": 49816, "LUT_pct": 93.6, "FF": 43492,
                  "BRAM18": 29, "BRAM18_pct": 10.4, "DSP": 22, "DSP_pct": 10.0},
    "clock_mhz": 50,
}

with open("results_10k.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 64)
print(" VGG-Lite accelerator on PYNQ-Z2 — results summary")
print("=" * 64)
print(f" images evaluated (accuracy): {N:,}")
print(f" clock frequency            : 50 MHz")
print("-" * 64)
print(f" HW/SW feature equivalence  : {exact:.4f} %   ({n_eq:,} images)")
print(f" End-to-end accuracy        : {acc_hw:.2f} %")
print("-" * 64)
print(f" Accelerator latency        : {t_accel:.2f} ms")
print(f" System latency             : {t_system:.2f} ms")
print(f"   speed-up, same layers    : x{t_middle/t_accel:.2f}")
print(f"   speed-up, system         : x{t_sw_all/t_system:.2f}")
print("-" * 64)
print(f" Accelerator throughput     : {thr_accel:.2f} ms/img ({1000/thr_accel:.1f} img/s)")
print(f" System throughput          : {thr_system:.2f} ms/img ({1000/thr_system:.1f} img/s)")
print(f" Pipeline efficiency        : {100*THEORETICAL_MS/thr_accel:.1f} %")
print(f" Effective compute rate     : {2*MACS_HW/(thr_accel/1000)/1e9:.2f} GOPS")
print("-" * 64)
print(f" MACs in hardware           : {MACS_HW:,} / {MACS_TOTAL:,} ({100*MACS_HW/MACS_TOTAL:.1f} %)")
print(f" Amdahl ceiling             : x{1/(1-MACS_HW/MACS_TOTAL):.2f}")
print("-" * 64)
print(f" LUT  : 49,816 / 53,200  (93.6 %)")
print(f" BRAM18: 29 / 280        (10.4 %)")
print(f" DSP  : 22 / 220         (10.0 %)")
print("=" * 64)
print("\nwritten: results_10k.json")

 VGG-Lite accelerator on PYNQ-Z2 — results summary
 images evaluated (accuracy): 10,000
 clock frequency            : 50 MHz
----------------------------------------------------------------
 HW/SW feature equivalence  : 100.0000 %   (1,000 images)
 End-to-end accuracy        : 70.29 %
----------------------------------------------------------------
 Accelerator latency        : 11.05 ms
 System latency             : 40.00 ms
   speed-up, same layers    : x6.82
   speed-up, system         : x2.45
----------------------------------------------------------------
 Accelerator throughput     : 8.15 ms/img (122.7 img/s)
 System throughput          : 19.59 ms/img (51.1 img/s)
 Pipeline efficiency        : 96.5 %
 Effective compute rate     : 1.45 GOPS
----------------------------------------------------------------
 MACs in hardware           : 5,898,240 / 6,341,888 (93.0 %)
 Amdahl ceiling             : x14.29
----------------------------------------------------------------
 LUT  : 49,816 / 